# 03 - 比较真实和生成细胞

比较 PerturbNet 模型生成的扰动后细胞与真实实验观测细胞的表达分布。

注意: 模型生成细胞不是真实实验观测，仅用于评估模型生成质量。

In [ ]:
import sys
sys.path.insert(0, '..')
from pathlib import Path
import anndata as ad
import numpy as np
import pandas as pd

# 查找数据
all_h5ad = []
for d in [Path('../data/example'), Path('../data/processed'), Path('../data/raw')]:
    if d.exists():
        all_h5ad.extend(list(d.glob('**/*.h5ad')))

print(f'Available files: {[f.name for f in all_h5ad]}')

# 尝试区分真实数据和生成数据
real_files = [f for f in all_h5ad if 'generated' not in f.name.lower() and 'synthetic' not in f.name.lower()]
gen_files = [f for f in all_h5ad if 'generated' in f.name.lower() or 'synthetic' in f.name.lower()]

print(f'\nReal data files: {[f.name for f in real_files]}')
print(f'Generated data files: {[f.name for f in gen_files]}')

In [ ]:
# 加载真实数据和生成数据进行比较
if real_files and gen_files:
    real_adata = ad.read_h5ad(real_files[0])
    gen_adata = ad.read_h5ad(gen_files[0])
    
    print(f'Real data: {real_adata.shape}')
    print(f'Generated data: {gen_adata.shape}')
    print()
    
    # 比较基因重叠
    real_genes = set(real_adata.var_names)
    gen_genes = set(gen_adata.var_names)
    common_genes = real_genes & gen_genes
    print(f'Common genes: {len(common_genes)}')
    print(f'Real-only genes: {len(real_genes - gen_genes)}')
    print(f'Generated-only genes: {len(gen_genes - real_genes)}')
else:
    print('Need both real and generated data files for comparison.')
    print('Generated cells can be produced by running PerturbNet inference.')

In [ ]:
# 表达分布比较 (如果有两类数据)
if 'real_adata' in locals() and 'gen_adata' in locals():
    from scipy.sparse import issparse
    
    common = list(common_genes)[:1000]  # 限制基因数
    
    real_X = real_adata[:, common].X
    gen_X = gen_adata[:, common].X
    
    if issparse(real_X):
        real_data = real_X.data
        real_mean = np.asarray(real_X.mean(axis=0)).flatten()
    else:
        real_data = real_X.flatten()
        real_mean = real_X.mean(axis=0)
    
    if issparse(gen_X):
        gen_data = gen_X.data
        gen_mean = np.asarray(gen_X.mean(axis=0)).flatten()
    else:
        gen_data = gen_X.flatten()
        gen_mean = gen_X.mean(axis=0)
    
    print('Expression distribution comparison:')
    print(f'  Real: mean={real_data.mean():.4f}, std={real_data.std():.4f}')
    print(f'  Generated: mean={gen_data.mean():.4f}, std={gen_data.std():.4f}')
    print()
    
    # 基因均值相关性
    corr = np.corrcoef(real_mean, gen_mean)[0, 1]
    print(f'Gene-wise mean expression correlation: {corr:.4f}')
    print()
    print('Note: Generated cells are model outputs, not real experimental observations.')

## 注意事项

- 模型权重不是表达数据
- 模型预测结果也不是真实实验观测
- 生成细胞仅用于评估模型生成质量
- 真实实验数据来自 GEO/Hugging Face 上的作者处理后数据